In [ ]:
!pip install -q requests beautifulsoup4 cloudscraper ipywidgets

# romsfun — PSP ROM Downloader

Scrapes romsfun.com game pages and downloads the ISO/CSO files directly.

---

## Source

- **Site:** https://romsfun.com
- **Section:** PlayStation Portable (PSP)
- **How to find URLs:** browse to any game page (e.g. `romsfun.com/roms/playstation-portable/<game>.html`) and paste the page URL — not the download link, the game page itself.

## What it does

1. Takes a list of romsfun game page URLs from the input widget.
2. Scrapes each page to find the direct download link (`.iso`, `.zip`, `.7z`, `.rar`, `.cso`, `.pbp`).
3. Downloads each file to `/content/downloads/romsfun/` inside the Colab runtime.
4. Copies all downloaded files to `MyDrive/CollabMedia/downloads/romsfun/` on Google Drive.

## How to use

1. Run **Cell 1** (`pip install`) — installs dependencies.
2. Run **Cell 2** — shows the URL input box.
3. Paste your romsfun game page URLs in the box, one per line.
4. Run **Cell 3** — reads URLs, sets up scraper.
5. Run **Cell 4** — defines download functions (no output).
6. Run **Cell 5** — starts downloading. Shows progress per file.
7. Run **Cell 6** — mounts Drive and copies files to `CollabMedia/`.

## Known limitations

- Some pages serve the download link via JavaScript — the scraper won't find it and will print `WARN: no download URL found`. Check those pages manually.
- If you get a 403, the site may have detected the bot. Try re-running with a longer `time.sleep` delay.
- Colab disconnects after ~12 h of inactivity — copy to Drive (Cell 6) before then or files are lost.

## Output

```
MyDrive/
└── CollabMedia/
    └── downloads/
        └── romsfun/
            ├── game1.iso
            ├── game2.zip
            └── ...
```


In [ ]:
import ipywidgets as widgets
from IPython.display import display

url_box = widgets.Textarea(
    value="",
    placeholder="Paste romsfun.com page URLs here, one per line",
    layout=widgets.Layout(width="100%", height="260px"),
)
display(widgets.Label("romsfun page URLs (one per line):"), url_box)


In [ ]:
import cloudscraper
from bs4 import BeautifulSoup
import os, time, re
from urllib.parse import urljoin

PAGES = [u.strip() for u in url_box.value.splitlines() if u.strip()]
print(f"{len(PAGES)} URLs queued.")

OUT_DIR = "/content/downloads/romsfun"
os.makedirs(OUT_DIR, exist_ok=True)

scraper = cloudscraper.create_scraper()
scraper.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
})

In [ ]:
def find_download_url(page_url):
    resp = scraper.get(page_url, timeout=30)
    soup = BeautifulSoup(resp.text, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if re.search(r"\.(iso|zip|7z|rar|cso|pbp)(\?|$)", href, re.I):
            return urljoin(page_url, href)

    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "/download/" in href or "download" in a.get("class", []):
            return urljoin(page_url, href)

    for tag in soup.find_all(attrs={"data-url": True}):
        return tag["data-url"]
    for tag in soup.find_all(attrs={"data-href": True}):
        return tag["data-href"]

    return None


def download_file(url, out_dir):
    filename = url.split("/")[-1].split("?")[0]
    out_path = os.path.join(out_dir, filename)

    if os.path.exists(out_path):
        print(f"  SKIP (exists): {filename}")
        return

    print(f"  Downloading: {filename}")
    with scraper.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                f.write(chunk)
                downloaded += len(chunk)
                if total:
                    pct = downloaded / total * 100
                    print(f"\r  {pct:.1f}% ({downloaded/1024/1024:.1f} MB)", end="")
    print(f"\n  Done: {out_path}")

In [ ]:
failed = []

for page in PAGES:
    name = page.split("/")[-1].replace(".html", "")
    print(f"\n[{name}]")
    try:
        dl_url = find_download_url(page)
        if not dl_url:
            print("  WARN: no download URL found — check page manually")
            print(f"  Page: {page}")
            failed.append(page)
            continue
        print(f"  URL: {dl_url}")
        download_file(dl_url, OUT_DIR)
    except Exception as e:
        print(f"  ERROR: {e}")
        failed.append(page)
    time.sleep(2)

print("\n=== DONE ===")
if failed:
    print("Failed (check manually):")
    for p in failed:
        print(" ", p)
else:
    print("All succeeded.")

In [ ]:
# Copy output to Google Drive under CollabMedia/downloads/romsfun/
from google.colab import drive
import shutil

drive.mount("/content/drive")

DRIVE_OUT = "/content/drive/MyDrive/CollabMedia/downloads/romsfun"
os.makedirs(DRIVE_OUT, exist_ok=True)

for f in os.listdir(OUT_DIR):
    src = os.path.join(OUT_DIR, f)
    dst = os.path.join(DRIVE_OUT, f)
    print(f"Copying {f} → Drive")
    shutil.copy2(src, dst)

print(f"Done. Files at: {DRIVE_OUT}")